# 07MIAR - Redes Neuronales y Deep Learning: Proyecto de programación "*Deep Vision in classification tasks*"


## Nombre y apellidos

* Rosa Chamorro Nieto
* Pablo Fenollera
* Iván Villanueva


## Enunciado

En esta actividad, el alumno debe **evaluar y comparar dos estrategias** para la **clasificación de imágenes** empleando el **dataset asignado**. Los alumnos deberá resolver el reto proponiendo una solución válida **basada en aprendizaje profundo**, más concretamente en redes neuronales convolucionales (**CNNs**). Será indispensable que la solución propuesta siga el **pipeline visto en clase** para resolver este tipo de tareas de inteligencia artificial:

1.   **Carga** del conjunto de datos
2.   **Inspección** del conjunto de datos
3.   **Acondicionamiento** del conjunto de datos
4.   Desarrollo de la **arquitectura** de red neuronal y **entrenamiento** de la solución
5.   **Monitorización** del proceso de **entrenamiento** para la toma de decisiones
6.   **Evaluación** del modelo predictivo y planteamiento de la siguiente prueba experimental

### Estrategia 1: Entrenar desde cero o *from scratch*

La primera estrategia a comparar será una **red neuronal profunda** que el **alumno debe diseñar, entrenar y optimizar**. Se debe **justificar empíricamente** las decisiones que llevaron a la selección de la **arquitectura e hiperparámetros final**. Se espera que el alumno utilice todas las **técnicas de regularización** mostradas en clase de forma justificada para la mejora del rendimiento de la red neuronal (*weight regularization*, *dropout*, *batch normalization*, *data augmentation*, etc.).

### Estrategia 2: Red pre-entrenada

La segunda estrategia a comparar debe incluir la utilización de una **red preentrenada** con el dataset ImageNet, llevando a cabo tareas de ***transfer learning*** y ***fine-tuning*** para resolver la tarea de clasificación asignada. Deben **compararse al menos dos tipos de arquitecturas** (VGGs, ResNet50, Xception, InceptionV3, InceptionResNetV2, MobileNetV2, DenseNet, ResNet) y se debe **seleccionar la que mayor precisión proporcione** (información sobre las arquitecturas disponibles en https://keras.io/applications/). Se espera el uso de todas las **técnicas de regularización** mostradas en clase de forma justificada para la mejora del rendimiento de la red neuronal (*weight regularization*, *dropout*, *batch normalization*, *data augmentation*, etc.).

## Normas a seguir

- Será **indispensable** realizar el **trabajo por parejas**. Dichas parejas de alumnxs se generarán **de manera automática** teniendo en cuenta el pais de residencia con el objetivo de facilitar el trabajo en equipo.  
- Se debe entregar un **ÚNICO FICHERO PDF POR ALUMNO** que incluya las instrucciones presentes en el Colab Noteboook y su **EJECUCIÓN!!!**. Debe aparecer todo el proceso llevado a cabo en cada estrategia (i.e. carga de datos, inspección de datos, acondicionamiento, proceso de entrenamiento y proceso de validación del modelo).
- **La memoria del trabajo** (el fichero PDF mencionado en el punto anterior) deberá **subirla cada integrante del grupo** (aunque se trate de un documento idéntico) a la actividad que se habilitará **en CampusVIU**.
- Se recomienda trabajar respecto a un directorio base (**BASE_FOLDER**) para facilitar el trabajo en equipo. En este notebook se incluye un ejemplo de cómo almacenar/cargar datos utilizando un directorio base.
- Las **redes propuestas** deben estar **entrenadas** (y **EVIDENCIAR este proceso en el documento PDF**). La entrega de una **red sin entrenar** supondrá **perdida de puntos**.
- Si se desea **evidenciar alguna métrica** del proceso de entrenamiento (precisión, pérdida, etc.), estas deben ser generadas.
- Todos los **gráficos** que se deseen mostrar deberán **generarse en el Colab Notebook** para que tras la conversión aparezcan en el documento PDF.

## Librerías

### Instalacion de librerias necesarias para ejecutar el programa

In [1]:
!pip install --upgrade --force-reinstall --no-deps kaggle
!pip install tqdm
!pip install zipfile
!pip install tensorflow
!pip install dotenv
!pip install slugify
!pip install --upgrade "kagglesdk>=0.1.28"

  Using cached kaggle-2.2.1-py3-none-any.whl.metadata (16 kB)
Using cached kaggle-2.2.1-py3-none-any.whl (132 kB)
  Attempting uninstall: kaggle
    Found existing installation: kaggle 2.2.1
    Uninstalling kaggle-2.2.1:
      Successfully uninstalled kaggle-2.2.1



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: Could not find a version that satisfies the requirement zipfile (from versions: none)

[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for zipfile



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


  Attempting uninstall: kagglesdk
    Found existing installation: kagglesdk 0.1.28
    Uninstalling kagglesdk-0.1.28:
      Successfully uninstalled kagglesdk-0.1.28


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
kaggle 2.2.1 requires jupytext, which is not installed.
kaggle 2.2.1 requires python-slugify, which is not installed.

[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


### Imports

In [2]:
import zipfile
import os
from collections import Counter
from PIL import Image
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

## Preprocesado
### 1.   **Carga** del conjunto de datos

In [3]:
# Descarga del dataset
!kaggle datasets download -d paultimothymooney/chest-xray-pneumonia

# Descompresion del dataset
with zipfile.ZipFile("chest-xray-pneumonia.zip", "r") as zip_ref:
    zip_ref.extractall("chest_xray")

Dataset URL: https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia
License(s): other
chest-xray-pneumonia.zip: Skipping, found more recently modified local copy (use --force to force download)


### 2.   **Inspección** del conjunto de datos

In [4]:
# Ruta base donde están las carpetas train / val / test
# base_path = os.path.join(os.environ["USERPROFILE"], "Downloads", "chest_xray", "chest_xray", "chest_xray")
# TODO: Mejor Working Directory?
base_path = os.path.join(os.getcwd(), "chest_xray", "chest_xray", "chest_xray")
print(base_path)

# Carga del conjunto de entrenamiento
train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    os.path.join(base_path, "train"),
    image_size=(224, 224),
    batch_size=16,
    shuffle=True
)

# Carga del conjunto de validación
val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    os.path.join(base_path, "val"),
    image_size=(224, 224),
    batch_size=16
)

# Carga del conjunto de test
test_ds = tf.keras.preprocessing.image_dataset_from_directory(
    os.path.join(base_path, "test"),
    image_size=(224, 224),
    batch_size=16
)

# Muestra las clases detectadas automáticamente
num_classes = len(train_ds.class_names)
class_names = train_ds.class_names
print("Clases del dataset: ", class_names)

# Función para verificar si una imagen está dañada
def es_valida(ruta):
    try:
        Image.open(ruta).verify()
        return True
    except:
        return False

# Recorre train / val / test y elimina imágenes corruptas
for split in os.listdir(base_path):
    if not os.path.isfile(os.path.join(base_path, split)):
        for clase in class_names:
            carpeta = os.path.join(base_path, split, clase)
            for archivo in os.listdir(carpeta):
                ruta = os.path.join(carpeta, archivo)
                if not es_valida(ruta):
                    os.remove(ruta)
                    print("Imagen corrupta o archivo sobrante:", ruta)

# Conteo de imágenes por clase en cada partición
for split in os.listdir(base_path):
    if not os.path.isfile(os.path.join(base_path, split)):
        conteo = {clase: len(os.listdir(os.path.join(base_path, split, clase))) for clase in os.listdir(os.path.join(base_path, split)) if not os.path.isfile(os.path.join(base_path, clase))}
        print(f"Imágenes del conjunto de {split} por clase:", conteo)

c:\Users\rosai\Repositorios\proyecto_redes_neuronales_VIU\chest_xray\chest_xray\chest_xray
Found 5216 files belonging to 2 classes.
Found 16 files belonging to 2 classes.
Found 624 files belonging to 2 classes.
Clases del dataset:  ['NORMAL', 'PNEUMONIA']
Imagen corrupta o archivo sobrante: c:\Users\rosai\Repositorios\proyecto_redes_neuronales_VIU\chest_xray\chest_xray\chest_xray\train\NORMAL\.DS_Store
Imagen corrupta o archivo sobrante: c:\Users\rosai\Repositorios\proyecto_redes_neuronales_VIU\chest_xray\chest_xray\chest_xray\train\PNEUMONIA\.DS_Store
Imagen corrupta o archivo sobrante: c:\Users\rosai\Repositorios\proyecto_redes_neuronales_VIU\chest_xray\chest_xray\chest_xray\val\NORMAL\.DS_Store
Imagen corrupta o archivo sobrante: c:\Users\rosai\Repositorios\proyecto_redes_neuronales_VIU\chest_xray\chest_xray\chest_xray\val\PNEUMONIA\.DS_Store
Imágenes del conjunto de test por clase: {'NORMAL': 234, 'PNEUMONIA': 390}
Imágenes del conjunto de train por clase: {'NORMAL': 1341, 'PNEUMON

### 3.   **Acondicionamiento** del conjunto de datos

In [5]:
# Normalización: escala los píxeles de [0,255] a [0,1]
normalization = tf.keras.layers.Rescaling(1./255)
train_ds = train_ds.map(lambda x, y: (normalization(x), y))
val_ds = val_ds.map(lambda x, y: (normalization(x), y))
test_ds = test_ds.map(lambda x, y: (normalization(x), y))

# Data augmentation: aumenta variabilidad del dataset
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

# Aplica augmentation solo al conjunto de entrenamiento
train_ds = train_ds.map(
    lambda x, y: (data_augmentation(x, training=True), y)
)

# Optimización
train_ds = train_ds.cache().prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.cache().prefetch(tf.data.AUTOTUNE)
test_ds = test_ds.cache().prefetch(tf.data.AUTOTUNE)

## Estrategia 1: CNN from scratch

### Definición del modelo

Justificación Empírica de la Arquitectura e Hiperparámetros

Para llegar a la arquitectura y configuración de hiperparámetros final de nuestra estrategia from scratch, se llevó a cabo un proceso iterativo de experimentación empírica. El principal reto de este dataset (imágenes médicas de rayos X) es su alta tendencia al sobreajuste (overfitting), debido a la gran similitud visual entre las radiografías y el desbalance natural de las clases.

A continuación, justificamos las decisiones tomadas en base a las observaciones experimentales:

1. Evolución de la Arquitectura:

* Modelo Base Inicial: Inicialmente, probamos una red sencilla de solo 2 bloques convolucionales sin técnicas de regularización. Observamos empíricamente que la red memorizaba rápidamente el conjunto de entrenamiento (Accuracy de train > 95% en las primeras épocas), pero la curva de validación se estancaba y la pérdida (Loss) de validación comenzaba a subir dramáticamente.

* Profundidad de la red (3 Bloques Conv): Para extraer características más complejas (diferenciar la opacidad pulmonar de la neumonía vs un pulmón sano), aumentamos la profundidad a 3 bloques convolucionales, doblando el número de filtros secuencialmente (32 $\rightarrow$ 64 $\rightarrow$ 128). Esta estructura de "embudo" es el estándar empírico que mejor equilibrio nos ofreció entre capacidad de aprendizaje y coste computacional.

2. Selección de Técnicas de Regularización (Mitigación del Overfitting):
Dado el sobreajuste inicial, fuimos incorporando las siguientes técnicas y observando su impacto:

* Dropout (0.5): Al introducir una capa densa de 512 neuronas, la red tendía a depender de píxeles muy específicos. Al aplicar un Dropout severo del 50% justo antes de la salida, forzamos a la red a aprender representaciones redundantes y más robustas, lo que redujo drásticamente la brecha entre el Accuracy de entrenamiento y validación.

* Batch Normalization: Notamos que al entrenar, las gráficas de Loss tenían muchos "picos" y eran inestables. Al añadir Batch Normalization después de cada capa convolucional, logramos estabilizar los gradientes, consiguiendo curvas de entrenamiento mucho más suaves y una convergencia más rápida.

* Regularización L2 (Weight Decay = 0.001): Observamos que los pesos de las capas densas crecían demasiado. Aplicar una penalización L2 ayudó a mantener los pesos pequeños, limitando aún más la capacidad de la red para sobreajustarse al ruido del dataset.

3. Hiperparámetros y Monitorización:

* Learning Rate (0.001): Se utilizó el optimizador Adam con su tasa de aprendizaje por defecto. Pruebas con un learning rate mayor (ej. 0.01) hacían que el modelo no lograra converger, mientras que valores menores (0.0001) hacían el entrenamiento inviable por su lentitud.

* Early Stopping (Patience = 5): En lugar de fijar un número estricto de épocas y arriesgarnos a sobreajustar en las fases finales, implementamos un Early Stopping. Empíricamente, comprobamos que tras unas 10-15 épocas, el modelo dejaba de aprender generalizaciones útiles. El Early Stopping nos aseguró quedarnos con la versión de los pesos que minimiza la pérdida de validación.

#### Primer modelo

In [14]:
def primer_modelo(input_shape=(224, 224, 3)):
    """
    Crea y compila la arquitectura CNN desde cero.
    """
    model = tf.keras.Sequential([
        layers.Input(input_shape),

        # Bloque convolucional 1
        layers.Conv2D(32, (3,3), activation="relu"),
        layers.MaxPooling2D(),

        # Bloque convolucional 2
        layers.Conv2D(64, (3,3), activation="relu"),
        layers.MaxPooling2D(),

        # Clasificación
        layers.Flatten(),
        layers.Dense(64, activation="relu"),
        # Salida binaria
        layers.Dense(1, activation="sigmoid")
    ])

    return model

# 2. Instanciar el modelo
primer_modelo = primer_modelo()
primer_modelo.summary() # Muestra la arquitectura


Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_12 (Conv2D)              │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_12 (MaxPooling2D) │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_13 (Conv2D)              │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_13 (MaxPooling2D) │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_4 (Flatten)             │ (None, 186624)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 64)             │    11,944,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,963,457 (45.64 MB)

 Trainable params: 11,963,457 (45.64 MB)

 Non-trainable params: 0 (0.00 B)

#### Modelo mejorado

In [15]:
def modelo_cnn_mejorado(input_shape=(224, 224, 3)):
    """
    Crea y compila la arquitectura CNN desde cero.
    """
    modelo = models.Sequential([
        # Bloque Convolucional 1
        layers.Conv2D(32, (3, 3), padding='same', input_shape=input_shape),
        layers.BatchNormalization(), # Regularización: Batch Normalization
        layers.Activation('relu'),
        layers.MaxPooling2D((2, 2)),
        
        # Bloque Convolucional 2
        layers.Conv2D(64, (3, 3), padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D((2, 2)),
        
        # Bloque Convolucional 3
        layers.Conv2D(128, (3, 3), padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D((2, 2)),
        
        # Aplanado y Capas Densas
        layers.Flatten(),
        # Regularización L2 (Weight Decay) aplicada a los pesos de la capa densa
        layers.Dense(512, kernel_regularizer=regularizers.l2(0.001)),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.Dropout(0.5), # Regularización: Dropout para evitar sobreajuste
        
        # Capa de salida
        # Asumiendo clasificación binaria: 1 neurona con función Sigmoide
        # Si las etiquetas están codificadas diferente, esto podría cambiar a Dense(2, activation='softmax')
        layers.Dense(1, activation='sigmoid')
    ])
    
    return modelo

# 2. Instanciar el modelo
modelo_cnn_mejorado = modelo_cnn_mejorado()
modelo_cnn_mejorado.summary() # Muestra la arquitectura

Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_14 (Conv2D)              │ (None, 224, 224, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_8           │ (None, 224, 224, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_8 (Activation)       │ (None, 224, 224, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_14 (MaxPooling2D) │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_15 (Conv2D)              │ (None, 112, 112, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_9           │ (None, 112, 112, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_9 (Activation)       │ (None, 112, 112, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_15 (MaxPooling2D) │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_16 (Conv2D)              │ (None, 56, 56, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_10          │ (None, 56, 56, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_10 (Activation)      │ (None, 56, 56, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_16 (MaxPooling2D) │ (None, 28, 28, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_5 (Flatten)             │ (None, 100352)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 512)            │    51,380,736 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_11          │ (None, 512)            │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_11 (Activation)      │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 1)              │           513 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 51,477,441 (196.37 MB)

 Trainable params: 51,475,969 (196.37 MB)

 Non-trainable params: 1,472 (5.75 KB)

### Entrenamiento

In [16]:
def entrenar_modelo(modelo, train_ds, val_ds, epochs=15, learning_rate=0.001):
    # Compilación del modelo
    modelo.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss='binary_crossentropy', # Función de pérdida para clasificación binaria
        metrics=['accuracy']
    )
    
    # Callback: Early Stopping
    # Detiene el entrenamiento si la pérdida de validación no mejora tras 'patience' épocas
    # y restaura los mejores pesos encontrados.
    early_stopping = tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=1
    )
    
    print("Iniciando el entrenamiento...")
    history = modelo.fit(
        train_ds,
        validation_data=val_ds,
        epochs=epochs,
        callbacks=[early_stopping]
    )
    return modelo, history

#### Primer modelo

In [ ]:
# 3. Entrenar
modelo_entrenado, historial = entrenar_modelo(
    primer_modelo, train_ds, val_ds, epochs=15
    )

Iniciando el entrenamiento...
Epoch 1/15


#### Modelo mejorado

In [ ]:
# 3. Entrenar
modelo_entrenado_mejorado, historial_mejorado = entrenar_modelo(modelo_cnn_mejorado, train_ds, val_ds, epochs=15)

### Monitorización

In [ ]:
def graficar_monitorizacion(history):
    """
    Genera gráficas de la evolución del Loss y el Accuracy.
    """
    acc = history.history['accuracy']
    val_acc = history.history['val_accuracy']
    loss = history.history['loss']
    val_loss = history.history['val_loss']
    epochs_range = range(1, len(acc) + 1)
    
    plt.figure(figsize=(12, 5))
    
    # Gráfica de Pérdida
    plt.subplot(1, 2, 1)
    plt.plot(epochs_range, loss, 'b-', label='Pérdida de Entrenamiento')
    plt.plot(epochs_range, val_loss, 'r-', label='Pérdida de Validación')
    plt.title('Pérdida (Loss) durante el Entrenamiento')
    plt.xlabel('Épocas')
    plt.ylabel('Pérdida')
    plt.legend(loc='upper right')
    
    # Gráfica de Exactitud
    plt.subplot(1, 2, 2)
    plt.plot(epochs_range, acc, 'b-', label='Exactitud de Entrenamiento')
    plt.plot(epochs_range, val_acc, 'r-', label='Exactitud de Validación')
    plt.title('Exactitud (Accuracy) durante el Entrenamiento')
    plt.xlabel('Épocas')
    plt.ylabel('Exactitud')
    plt.legend(loc='lower right')
    
    plt.tight_layout()
    plt.show()

#### Primer modelo

In [ ]:
# 4. Monitorizar
graficar_monitorizacion(historial)

#### Modelo mejorado

In [ ]:
# 4. Monitorizar
graficar_monitorizacion(historial_mejorado)

### Evaluación

In [ ]:
def evaluar_modelo(modelo, test_ds):
    print("\n--- EVALUACIÓN EN CONJUNTO DE TEST ---")
    test_loss, test_acc = modelo.evaluate(test_ds)
    print(f"Test Loss: {test_loss:.4f}")
    print(f"Test Accuracy: {test_acc:.4f}\n")
    
    # Extraer etiquetas reales iterando sobre el dataset de test
    etiquetas_reales = np.concatenate([y.numpy() for x, y in test_ds], axis=0)
    
    # Predecir sobre todo el dataset de test
    preds = modelo.predict(test_ds)
    
    # Convertir probabilidades a clases binarias (0 o 1) usando umbral de 0.5
    predicciones = (preds > 0.5).astype(int).flatten()
    
    print("--- REPORTE DE CLASIFICACIÓN ---")
    nombres_clases = ['Normal', 'Neumonía'] # Verifica que este orden coincida con los directorios de Kaggle
    print(classification_report(etiquetas_reales, predicciones, target_names=nombres_clases))
    
    # Matriz de Confusión
    cm = confusion_matrix(etiquetas_reales, predicciones)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=nombres_clases, yticklabels=nombres_clases)
    plt.title('Matriz de Confusión - Test')
    plt.ylabel('Verdadero')
    plt.xlabel('Predicción')
    plt.show()

#### Primer modelo

In [ ]:
# 5. Evaluar
evaluar_modelo(modelo_entrenado, test_ds)

#### Modelo mejorado

In [ ]:
# 5. Evaluar
evaluar_modelo(modelo_entrenado_mejorado, test_ds)